# Práctica 2. Resolución de Problemas con Búsqueda
## Ingeniería del Conocimiento 2025/2026
### Prof. Juan A. Recio García

---
**Objetivo:** representar problemas clásicos de IA como espacios de estados y resolverlos con algoritmos de búsqueda ciega de la librería AIMA.


## Instalación e importación
Si falta `numpy` instálalo con la siguiente celda:

In [3]:
%pip install numpy -q
from search import *
print('Librería AIMA cargada correctamente ✅')

Note: you may need to restart the kernel to use updated packages.
Librería AIMA cargada correctamente ✅



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## La clase `Problem` (AIMA)

Todo problema concreto es una **subclase de `Problem`** que implementa:

| Método | Descripción |
|---|---|
| `__init__(initial, goal)` | Estado inicial y objetivo |
| `actions(state)` | Lista de acciones aplicables en `state` |
| `result(state, action)` | Estado resultante de aplicar la acción |
| `goal_test(state)` | `True` si `state` es objetivo |
| `path_cost(c, s1, a, s2)` | Coste acumulado (por defecto +1 por paso) |
| `coste_de_aplicar_accion(s, a)` | Coste de UN solo operador |

La **estrategia de búsqueda** decide qué nodo expandir a continuación. Los nodos no expandidos se guardan en la **frontera** (abiertos).


---
# EJERCICIO 1: Misioneros y Caníbales

## Descripción
3 misioneros y 3 caníbales deben cruzar un río. La barca lleva máximo **2 personas**.  
**Restricción:** en ninguna orilla puede haber más caníbales que misioneros (salvo que no haya misioneros).

## Estado: tupla `(M, C, B)`
- `M` → misioneros en la orilla **inicial**
- `C` → caníbales en la orilla **inicial**
- `B` → bote (`0`=orilla inicial, `1`=orilla final)

**Inicial:** `(3,3,0)` → **Objetivo:** `(0,0,1)`

## Acciones
| Acción | Significado |
|---|---|
| `('1c', (0,1))` | Mueve 1 caníbal |
| `('1m', (1,0))` | Mueve 1 misionero |
| `('2c', (0,2))` | Mueve 2 caníbales |
| `('2m', (2,0))` | Mueve 2 misioneros |
| `('1m1c', (1,1))` | Mueve 1 misionero + 1 caníbal |


In [4]:
class ProblemaMisioneros(Problem):
    '''Formalización del problema de misioneros y caníbales.'''

    def __init__(self, initial, goal=None):
        Problem.__init__(self, initial, goal)
        # (nombre_accion, (misioneros_en_barca, canibales_en_barca))
        self._actions = [
            ('1c',   (0, 1)),
            ('1m',   (1, 0)),
            ('2c',   (0, 2)),
            ('2m',   (2, 0)),
            ('1m1c', (1, 1))
        ]

    def actions(self, s):
        '''Solo devuelve acciones que llevan a estados válidos.'''
        return [a for a in self._actions if self._is_valid(self.result(s, a))]

    def _is_valid(self, s):
        '''
        Estado válido si:
        - Cantidades entre 0 y 3
        - En cada orilla: misioneros >= caníbales (salvo que no haya misioneros)
        '''
        M, C, _ = s
        return (
            0 <= M <= 3 and 0 <= C <= 3 and
            (M >= C or M == 0) and
            ((3-M) >= (3-C) or (3-M) == 0)
        )

    def result(self, s, a):
        '''Mueve personas según la dirección del bote.'''
        if s[2] == 0:  # bote va hacia la orilla final
            return (s[0] - a[1][0], s[1] - a[1][1], 1)
        else:          # bote vuelve a la orilla inicial
            return (s[0] + a[1][0], s[1] + a[1][1], 0)


### Pruebas básicas del problema

In [5]:
misioneros = ProblemaMisioneros((3, 3, 0), (0, 0, 1))

print("Estado inicial:", misioneros.initial)
print("¿Es objetivo el estado inicial?:", misioneros.goal_test(misioneros.initial))
print("¿Es objetivo (0,0,1)?:", misioneros.goal_test((0, 0, 1)))
print("Acciones en estado inicial:", misioneros.actions(misioneros.initial))
print("Resultado de mover 2 caníbales:", misioneros.result(misioneros.initial, ('2c', (0, 2))))


Estado inicial: (3, 3, 0)
¿Es objetivo el estado inicial?: False
¿Es objetivo (0,0,1)?: True
Acciones en estado inicial: [('1c', (0, 1)), ('2c', (0, 2)), ('1m1c', (1, 1))]
Resultado de mover 2 caníbales: (3, 1, 1)


### Búsqueda en Anchura (BFS) — `breadth_first_tree_search`
Explora **nivel por nivel** con una cola FIFO. Completa y óptima para costes iguales.

In [6]:
sol = breadth_first_tree_search(misioneros)
print("BFS — Solución:", sol.solution())


BFS — Solución: [('2c', (0, 2)), ('1c', (0, 1)), ('2c', (0, 2)), ('1c', (0, 1)), ('2m', (2, 0)), ('1m1c', (1, 1)), ('2m', (2, 0)), ('1c', (0, 1)), ('2c', (0, 2)), ('1c', (0, 1)), ('2c', (0, 2))]


### Búsqueda en Profundidad (DFS) — PROBLEMA CON CICLOS
⚠️ `depth_first_tree_search` **no tiene control de estados repetidos** → entra en **bucle infinito** con este problema.

In [7]:
# NO ejecutar depth_first_tree_search directamente: entra en bucle infinito
# porque el espacio de misioneros tiene ciclos:
# (3,3,0) → mover 1c → (3,2,1) → devolver 1c → (3,3,0) → ... ♾️
print("depth_first_tree_search NO se ejecuta por seguridad.")
print("Usar depth_first_GRAPH_search que controla estados ya visitados:")
sol_dfs = depth_first_graph_search(misioneros)
print("DFS (graph) — Solución:", sol_dfs.solution())


depth_first_tree_search NO se ejecuta por seguridad.
Usar depth_first_GRAPH_search que controla estados ya visitados:
DFS (graph) — Solución: [('1m1c', (1, 1)), ('1m', (1, 0)), ('2c', (0, 2)), ('1c', (0, 1)), ('2m', (2, 0)), ('1m1c', (1, 1)), ('2m', (2, 0)), ('1c', (0, 1)), ('2c', (0, 2)), ('1m', (1, 0)), ('1m1c', (1, 1))]


### ✏️ EJERCICIO 1A: ¿Por qué no funciona `depth_first_tree_search`?

**Respuesta:**  
`depth_first_tree_search` es una búsqueda **en árbol** — no guarda los estados ya visitados. El problema de los Misioneros tiene **ciclos** (se puede volver al mismo estado aplicando la acción inversa). DFS sigue ese ciclo infinitamente sin detectarlo.

**Ejemplo de ciclo:**
```
(3,3,0) → mover 1 caníbal → (3,2,1) → volver 1 caníbal → (3,3,0) → ...
```

**Solución:** usar `depth_first_graph_search`, que mantiene una lista de **nodos cerrados** (ya expandidos) y evita volver a visitarlos.


### Búsqueda de Coste Uniforme (UCS)

In [8]:
sol_ucs = uniform_cost_search(misioneros)
print("UCS — Solución:", sol_ucs.solution())


UCS — Solución: [('1m1c', (1, 1)), ('1m', (1, 0)), ('2c', (0, 2)), ('1c', (0, 1)), ('2m', (2, 0)), ('1m1c', (1, 1)), ('2m', (2, 0)), ('1c', (0, 1)), ('2c', (0, 2)), ('1c', (0, 1)), ('2c', (0, 2))]


### Búsqueda de Profundidad Limitada — PROBLEMA

In [9]:
resultado = depth_limited_search(misioneros, 10)
print("Profundidad limitada (L=10):", resultado)
# Devuelve 'cutoff' o None porque:
# 1. La solución real necesita 11 pasos (L=10 es insuficiente)
# 2. Al ser tree_search, genera ciclos antes de llegar


Profundidad limitada (L=10): cutoff


### ✏️ EJERCICIO 1B: ¿Por qué no funciona `depth_limited_search` con L=10?

**Respuesta:**  
Dos motivos combinados:
1. La solución óptima al problema de los Misioneros requiere **11 movimientos** mínimo. Con `L=10` el objetivo nunca se alcanza → devuelve `'cutoff'`.
2. Aunque se usase `L=11`, la implementación de AIMA es **tree search** (sin control de repetidos), por lo que genera ciclos que llenan la pila antes de alcanzar profundidad 11.

**Solución:** aumentar L a 11 Y usar búsqueda en grafo, o directamente usar `iterative_deepening_search`.


---
# EJERCICIO 2: El 8-Puzzle

## Descripción
Tablero 3×3 con fichas 1–8 y un hueco (`0`). Objetivo: llegar a la configuración ordenada.

```
Inicial        Goal
| 2 | 3 | 8 |  | 1 | 2 | 3 |
| 1 | 6 | 4 |  | 4 | 5 | 6 |
| 7 | 0 | 5 |  | 7 | 8 | 0 |
```

## Estado: tupla de 9 elementos (posiciones 0–8, leídas por filas)
```
pos: 0 | 1 | 2
     3 | 4 | 5
     6 | 7 | 8
```

## Cuándo es válida cada acción
| Acción | Condición |
|---|---|
| Mover hueco **arriba** | `pos_hueco >= 3` (no está en fila superior) |
| Mover hueco **abajo** | `pos_hueco <= 5` (no está en fila inferior) |
| Mover hueco **izquierda** | `pos_hueco % 3 != 0` (no está en columna izquierda) |
| Mover hueco **derecha** | `pos_hueco % 3 != 2` (no está en columna derecha) |

## Cómo funciona `swap(x)`
El hueco intercambia con la ficha en `posición + x`:
- Arriba: `x = -3` (la fila de arriba está 3 posiciones antes)
- Abajo: `x = +3`
- Izquierda: `x = -1`
- Derecha: `x = +1`


In [10]:
class Ocho_Puzzle(Problem):
    """
    8-Puzzle: tablero 3x3, el 0 es el hueco.
    Estado: tupla de 9 elementos (permutación de 0-8).
    """

    def __init__(self, initial, goal=(1, 2, 3, 4, 5, 6, 7, 8, 0)):
        self.goal = goal
        Problem.__init__(self, initial, goal)

    def actions(self, estado):
        """Devuelve las acciones válidas según la posición del hueco."""
        pos_hueco = estado.index(0)
        accs = []

        if pos_hueco >= 3:        # NO está en la fila superior
            accs.append("Mover hueco arriba")
        if pos_hueco <= 5:        # NO está en la fila inferior
            accs.append("Mover hueco abajo")
        if pos_hueco % 3 != 0:   # NO está en la columna izquierda
            accs.append("Mover hueco izquierda")
        if pos_hueco % 3 != 2:   # NO está en la columna derecha
            accs.append("Mover hueco derecha")

        return accs

    def result(self, estado, accion):
        """Intercambia el hueco con la ficha adyacente según la acción."""
        pos_hueco = estado.index(0)
        l = list(estado)  # convertimos a lista para modificar

        def swap(x):
            # intercambia posición del hueco con posición (hueco + x)
            l[pos_hueco], l[pos_hueco + x] = l[pos_hueco + x], l[pos_hueco]

        if accion == "Mover hueco arriba":
            swap(-3)   # la ficha de arriba está 3 posiciones antes
        elif accion == "Mover hueco abajo":
            swap(+3)   # la ficha de abajo está 3 posiciones después
        elif accion == "Mover hueco izquierda":
            swap(-1)
        elif accion == "Mover hueco derecha":
            swap(+1)

        return tuple(l)

    def h(self, node):
        """Heurística trivial = 1 (para búsqueda heurística, tema siguiente)."""
        return 1

    def coste_de_aplicar_accion(self, s, a):
        return 1


### Pruebas de la representación

In [11]:
p8 = Ocho_Puzzle((2, 3, 8, 1, 6, 4, 7, 0, 5))
print("Estado inicial:", p8.initial)
print()
e = p8.initial
for i in range(0, 9, 3):
    print(f"| {e[i]} | {e[i+1]} | {e[i+2]} |")
print()
print("Acciones posibles:", p8.actions(p8.initial))
print("  (Esperado: ['Mover hueco arriba', 'Mover hueco izquierda', 'Mover hueco derecha'])")


Estado inicial: (2, 3, 8, 1, 6, 4, 7, 0, 5)

| 2 | 3 | 8 |
| 1 | 6 | 4 |
| 7 | 0 | 5 |

Acciones posibles: ['Mover hueco arriba', 'Mover hueco izquierda', 'Mover hueco derecha']
  (Esperado: ['Mover hueco arriba', 'Mover hueco izquierda', 'Mover hueco derecha'])


In [12]:
# Aplicamos 'Mover hueco arriba'
nuevo = p8.result(p8.initial, "Mover hueco arriba")
print("Tras 'Mover hueco arriba':", nuevo)
for i in range(0, 9, 3):
    print(f"| {nuevo[i]} | {nuevo[i+1]} | {nuevo[i+2]} |")


Tras 'Mover hueco arriba': (2, 3, 8, 1, 0, 4, 7, 6, 5)
| 2 | 3 | 8 |
| 1 | 0 | 4 |
| 7 | 6 | 5 |


In [13]:
# 'Mover hueco abajo' desde (2,3,8,1,6,4,7,0,5)
# El hueco está en posición 7 (fila inferior) → actions() NO devuelve esta acción
# Si la llamamos directamente en result() → ERROR (swap intenta índice 10, fuera de rango)
try:
    p8.result(p8.initial, "Mover hueco abajo")
except IndexError as e:
    print("ERROR (esperado):", e)


ERROR (esperado): list index out of range


### ✏️ EJERCICIO 2A: ¿Por qué da error `result(estado, 'Mover hueco abajo')`?

**Respuesta:**  
El hueco está en la **posición 7** (fila inferior del tablero):
```
| 2 | 3 | 8 |
| 1 | 6 | 4 |
| 7 |[0]| 5 |  ← posición 7
```
Bajar significaría ir a `posición 7 + 3 = 10`, que **no existe** (solo hay posiciones 0–8).  
`swap(+3)` intenta `l[10]` → `IndexError: list index out of range`.

`actions()` ya lo previene (no incluye 'Mover hueco abajo' cuando `pos_hueco > 5`).  
Nunca se debe llamar a `result()` con una acción que `actions()` no haya devuelto.


### Aplicando los algoritmos de búsqueda

In [14]:
# Estado fácil (1-2 pasos al objetivo)
p8_facil = Ocho_Puzzle((1, 2, 3, 4, 5, 6, 7, 0, 8))
print("=== Estado fácil ===")
print("BFS :", breadth_first_graph_search(p8_facil).solution())
print("DFS :", depth_first_graph_search(p8_facil).solution())
print("UCS :", uniform_cost_search(p8_facil).solution())


=== Estado fácil ===
BFS : ['Mover hueco derecha']
DFS : ['Mover hueco derecha']
UCS : ['Mover hueco derecha']


In [15]:
# Estado más difícil
p8_dificil = Ocho_Puzzle((2, 3, 8, 1, 6, 4, 7, 0, 5))
print("=== Estado difícil ===")
print("Ejecutando BFS (graph)... puede tardar.")
sol = breadth_first_graph_search(p8_dificil)
print("BFS:", sol.solution())
print("Pasos:", len(sol.solution()))


=== Estado difícil ===
Ejecutando BFS (graph)... puede tardar.
BFS: ['Mover hueco arriba', 'Mover hueco derecha', 'Mover hueco arriba', 'Mover hueco izquierda', 'Mover hueco izquierda', 'Mover hueco abajo', 'Mover hueco derecha', 'Mover hueco abajo', 'Mover hueco derecha', 'Mover hueco arriba', 'Mover hueco izquierda', 'Mover hueco abajo', 'Mover hueco derecha']
Pasos: 13


### ✏️ EJERCICIO 2B: ¿Qué ocurre con los distintos métodos en el 8-Puzzle?

**Observaciones:**

| Algoritmo | Resultado | Motivo |
|---|---|---|
| `breadth_first_graph_search` | ✅ Solución óptima | Explora nivel a nivel con control de repetidos |
| `depth_first_graph_search` | ✅ Solución (no óptima) | Encuentra una solución pero puede ser larga |
| `depth_first_tree_search` | ❌ Bucle infinito | Sin control de repetidos → ciclos |
| `uniform_cost_search` | ✅ Óptima (=BFS) | Con costes=1 equivale a BFS |
| `depth_limited_search` | ⚠️ Solo si L≥profundidad real | Si L es pequeño devuelve 'cutoff' |

**Conclusión:** siempre usar **graph search** en problemas con ciclos.


---
# EJERCICIO 3: Granjero, Lobo, Cabra y Col

## Descripción
Cruzar un río con lobo, cabra y col. La barca lleva al **granjero + 1 cosa** máximo.

**Restricciones:**
- El lobo se come a la cabra si el granjero no está.
- La cabra se come la col si el granjero no está.

## Estado: `(granjero, lobo, cabra, col)`
- `False` = orilla izquierda
- `True` = orilla derecha

**Inicial:** `(False, False, False, False)` → **Objetivo:** `(True, True, True, True)`


In [16]:
class GranjeroLoboOvejaCol(Problem):
    """Granjero, lobo, cabra y col — cruzar el río."""

    def __init__(self):
        # Estado: (granjero, lobo, cabra, col) — False=izq, True=der
        Problem.__init__(self,
                         initial=(False, False, False, False),
                         goal=(True, True, True, True))

    def _is_valid(self, state):
        """
        Estado inválido si (sin el granjero):
        - lobo y cabra en la misma orilla → lobo se come a la cabra
        - cabra y col en la misma orilla → cabra se come la col
        """
        g, lobo, cabra, col = state
        # lobo y cabra juntos SIN granjero
        if lobo == cabra and g != lobo:
            return False
        # cabra y col juntas SIN granjero
        if cabra == col and g != cabra:
            return False
        return True

    def actions(self, state):
        """El granjero solo puede llevarse algo que esté en su orilla."""
        g, lobo, cabra, col = state
        posibles = ['solo']  # siempre puede cruzar solo

        if lobo == g:   posibles.append('lobo')
        if cabra == g:  posibles.append('cabra')
        if col == g:    posibles.append('col')

        # Filtrar por estados válidos resultantes
        return [a for a in posibles if self._is_valid(self.result(state, a))]

    def result(self, state, action):
        """El granjero (y opcionalmente algo) cambian de orilla."""
        g, lobo, cabra, col = state
        ng = not g  # el granjero siempre cruza

        if action == 'lobo':   return (ng, not lobo, cabra, col)
        if action == 'cabra':  return (ng, lobo, not cabra, col)
        if action == 'col':    return (ng, lobo, cabra, not col)
        return (ng, lobo, cabra, col)  # 'solo'


### Pruebas y búsquedas

In [17]:
problema_granjero = GranjeroLoboOvejaCol()
print("Estado inicial:", problema_granjero.initial)
print("Acciones posibles:", problema_granjero.actions(problema_granjero.initial))
# Solo 'cabra' y 'solo' son válidas desde el inicio


Estado inicial: (False, False, False, False)
Acciones posibles: ['cabra']


In [18]:
print("=== BFS tree ===")
print(breadth_first_tree_search(problema_granjero).solution())

print("\n=== BFS graph ===")
print(breadth_first_graph_search(problema_granjero).solution())

print("\n=== DFS graph ===")
print(depth_first_graph_search(problema_granjero).solution())

print("\n=== UCS ===")
print(uniform_cost_search(problema_granjero).solution())


=== BFS tree ===
['cabra', 'solo', 'lobo', 'cabra', 'col', 'solo', 'cabra']

=== BFS graph ===
['cabra', 'solo', 'lobo', 'cabra', 'col', 'solo', 'cabra']

=== DFS graph ===
['cabra', 'solo', 'col', 'cabra', 'lobo', 'solo', 'cabra']

=== UCS ===
['cabra', 'solo', 'col', 'cabra', 'lobo', 'solo', 'cabra']


### ✏️ Problemas encontrados

| Algoritmo | ¿Funciona? | Observación |
|---|---|---|
| `breadth_first_tree_search` | ✅ | Funciona (espacio pequeño, pocos ciclos antes del objetivo) |
| `depth_first_tree_search` | ❌ | Bucle infinito por ciclos |
| `breadth_first_graph_search` | ✅ | Óptimo — mínimo de pasos |
| `depth_first_graph_search` | ✅ | Funciona, no garantiza optimalidad |
| `uniform_cost_search` | ✅ | Óptimo (costes uniformes = BFS) |

**Conclusión:** `depth_first_tree_search` es la única variante que falla (ciclos). El resto resuelve el problema correctamente.


---
# EJERCICIO 4: Jarras de Agua

## Descripción
- Jarra grande: **4 litros**
- Jarra pequeña: **3 litros**

**Objetivo:** conseguir exactamente **2 litros** en alguna jarra.

## Estado: `(j4, j3)`
- `j4` ∈ [0,4] — litros en la jarra de 4L
- `j3` ∈ [0,3] — litros en la jarra de 3L

**Inicial:** `(0,0)` — **Objetivo:** `j4==2` OR `j3==2`

## 6 Acciones
| Acción | Efecto |
|---|---|
| `llenar_4` | j4 = 4 |
| `llenar_3` | j3 = 3 |
| `vaciar_4` | j4 = 0 |
| `vaciar_3` | j3 = 0 |
| `verter_4_a_3` | vierte j4→j3 hasta que j3 llena o j4 vacía |
| `verter_3_a_4` | vierte j3→j4 hasta que j4 llena o j3 vacía |


In [19]:
class JarrasDeAgua(Problem):
    """Dos jarras de 4L y 3L. Objetivo: tener 2L en alguna jarra."""

    def __init__(self):
        Problem.__init__(self, initial=(0, 0))
        self.cap4 = 4
        self.cap3 = 3

    def goal_test(self, state):
        """Objetivo múltiple: 2 litros en cualquiera de las dos jarras."""
        return state[0] == 2 or state[1] == 2

    def actions(self, state):
        """Solo devuelve acciones que tienen efecto real."""
        j4, j3 = state
        accs = []
        if j4 < self.cap4:              accs.append('llenar_4')
        if j3 < self.cap3:              accs.append('llenar_3')
        if j4 > 0:                      accs.append('vaciar_4')
        if j3 > 0:                      accs.append('vaciar_3')
        if j4 > 0 and j3 < self.cap3:  accs.append('verter_4_a_3')
        if j3 > 0 and j4 < self.cap4:  accs.append('verter_3_a_4')
        return accs

    def result(self, state, action):
        """Aplica la acción y devuelve el nuevo estado."""
        j4, j3 = state

        if action == 'llenar_4':      return (self.cap4, j3)
        if action == 'llenar_3':      return (j4, self.cap3)
        if action == 'vaciar_4':      return (0, j3)
        if action == 'vaciar_3':      return (j4, 0)
        if action == 'verter_4_a_3':
            # Vertemos lo que cabe en j3 (o todo lo que hay en j4)
            cant = min(j4, self.cap3 - j3)
            return (j4 - cant, j3 + cant)
        if action == 'verter_3_a_4':
            cant = min(j3, self.cap4 - j4)
            return (j4 + cant, j3 - cant)

    def coste_de_aplicar_accion(self, s, a):
        return 1


### Pruebas y búsquedas

In [20]:
jarras = JarrasDeAgua()
print("Estado inicial:", jarras.initial)
print("¿Es objetivo (0,0)?", jarras.goal_test((0, 0)))
print("¿Es objetivo (2,3)?", jarras.goal_test((2, 3)))
print("Acciones en (0,0):", jarras.actions((0, 0)))
print("Resultado de llenar_4:", jarras.result((0, 0), 'llenar_4'))


Estado inicial: (0, 0)
¿Es objetivo (0,0)? False
¿Es objetivo (2,3)? True
Acciones en (0,0): ['llenar_4', 'llenar_3']
Resultado de llenar_4: (4, 0)


In [21]:
print("=== BFS graph — Solución óptima ===")
sol_bfs = breadth_first_graph_search(jarras)
print("Solución:", sol_bfs.solution())
print("Pasos:", len(sol_bfs.solution()))


=== BFS graph — Solución óptima ===
Solución: ['llenar_3', 'verter_3_a_4', 'llenar_3', 'verter_3_a_4']
Pasos: 4


In [22]:
print("=== DFS graph ===")
sol_dfs = depth_first_graph_search(jarras)
print("Solución:", sol_dfs.solution())
print("Pasos:", len(sol_dfs.solution()))

print("\n=== UCS ===")
sol_ucs = uniform_cost_search(jarras)
print("Solución:", sol_ucs.solution())
print("Pasos:", len(sol_ucs.solution()))


=== DFS graph ===
Solución: ['llenar_3', 'verter_3_a_4', 'llenar_3', 'verter_3_a_4']
Pasos: 4

=== UCS ===
Solución: ['llenar_3', 'verter_3_a_4', 'llenar_3', 'verter_3_a_4']
Pasos: 4


### Traza manual de la solución BFS

```
(0,0) → llenar_4    → (4,0)
(4,0) → verter_4_a_3 → (1,3)
(1,3) → vaciar_3    → (1,0)
(1,0) → verter_4_a_3 → (0,1)
(0,1) → llenar_4    → (4,1)
(4,1) → verter_4_a_3 → (2,3)  ✅ j4 tiene exactamente 2 litros
```

BFS encuentra la solución en **6 pasos** (óptima). DFS puede encontrar un camino más largo.


---
# Resumen Final: Comparativa de Algoritmos

| Algoritmo | Completo | Óptimo | Tiempo | Espacio | ¿Controla ciclos? |
|---|---|---|---|---|---|
| BFS tree | ✅ | ✅ (costes=1) | O(r^p) | O(r^p) | ❌ |
| BFS graph | ✅ | ✅ (costes=1) | O(r^p) | O(r^p) | ✅ |
| DFS tree | ❌ | ❌ | O(r^m) | O(r·m) | ❌ |
| DFS graph | ✅ | ❌ | O(r^m) | O(r·m) | ✅ |
| UCS | ✅ | ✅ | O(r^C*/ε) | O(r^C*/ε) | ✅ |
| Depth-Limited | ✅ (L≥p) | ❌ | O(r^L) | O(r·L) | ❌ |

**Reglas prácticas:**
- Problema con ciclos → usar siempre **graph search**
- Necesitas solución óptima → **BFS** o **UCS**
- Poca memoria disponible → **DFS**
- No conoces la profundidad → **IDS** (profundización iterativa)
